# Silver layer

Preprocessing for text cleaning is as followed:  
- Normalize line endings
- Replace tabs and non-breaking spaces
- Remove zero-width spaces if present
- Remove trailing spaces on lines
- Remove repeated spaces inside lines
- Normalize spaces around line breaks
- Reduce too many blank lines
- Remove very common PDF page markers if present

In [88]:
# Import needed libraries
import pandas as pd
import re

# Import bronze dataframe + sanity checks

In [89]:
# Load bronze data
project_data = pd.read_json("../../Data/bronze/doc_02_bronze.json", orient='records')
print(project_data.head())


FileNotFoundError: File ../../Data/bronze/doc_02_bronze.json does not exist

# Clean incoming text data

First we will Normalize line endings

In [ ]:
def normalize_line_endings(text):
    return text.replace("\r\n", "\n").replace("\r", "\n")
#   print(normalize_line_endings(project_data['raw_text']))

Replace tabs and non-breaking spaces

In [ ]:
def replace_special_spaces(text):
    text = text.replace("\t", " ")
    text = text.replace("\xa0", " ")
    return text

Remove zero-width spaces if present

In [ ]:
def remove_zero_width_characters(text):
    zero_width_chars = ['\u200B', '\u200C', '\u200D', '\uFEFF']
    for char in zero_width_chars:
        text = text.replace(char, '')
    return text

Remove trailing spaces on lines

In [ ]:
def fix_hyphenated_linebreaks(text):
    # example: "infor-\nmation" -> "information"
    return re.sub(r'(\w)-\n(\w)', r'\1\2', text)

Remove repeated spaces inside lines

In [ ]:
def remove_pdf_page_markers(text):
    patterns = [
        r'Page\s+\d+\s+of\s+\d+',
        r'^\s*Page\s+\d+\s*$',
        r'^\s*\d+\s*\|\s*P\s*a\s*g\s*e\s*$',
        r'^\s*P\s*a\s*g\s*e\s*\d+\s*$',
        r'^\s*\d+\s*$'
    ]
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)
    return text

Normalize spaces around line breaks

In [ ]:
def remove_urls_and_emails(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', text)
    return text

Reduce too many blank lines

In [ ]:
def remove_trailing_spaces(text):
    return re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)


Remove very common PDF page markers if present

In [ ]:
def remove_repeated_spaces(text):
    return re.sub(r' {2,}', ' ', text)


In [ ]:
def normalize_linebreak_spacing(text):
    return re.sub(r' *\n *', '\n', text)

In [ ]:
def merge_broken_lines(text):
    lines = text.split('\n')
    merged = []
    
    for line in lines:
        line = line.strip()
        
        if not line:
            merged.append("")
            continue
        
        if not merged:
            merged.append(line)
            continue
        
        prev = merged[-1]
        
        # merge if previous line does not look like paragraph end
        if prev and not prev.endswith(('.', '!', '?', ':')) and line and not line[0].isupper():
            merged[-1] = prev + ' ' + line
        elif prev and not prev.endswith(('.', '!', '?', ':')) and len(line.split()) < 6:
            merged[-1] = prev + ' ' + line
        else:
            merged.append(line)
    
    return '\n'.join(merged)

In [ ]:
def remove_reference_like_lines(text):
    cleaned_lines = []
    
    for line in text.split('\n'):
        stripped = line.strip()
        
        # skip empty lines later
        if not stripped:
            cleaned_lines.append("")
            continue
        
        # very citation-heavy / bibliography-like patterns
        if re.match(r'^[A-Z][a-zA-Z\-]+,\s+[A-Z]\.', stripped):
            continue
        if re.search(r'\(\d{4}\)', stripped) and len(stripped.split()) < 12:
            continue
        if stripped.lower().startswith("references"):
            continue
        if stripped.lower().startswith("bibliography"):
            continue
        
        cleaned_lines.append(line)
    
    return '\n'.join(cleaned_lines)

In [ ]:
def remove_duplicate_lines(text):
    seen = set()
    result = []
    
    for line in text.split('\n'):
        key = re.sub(r'\s+', ' ', line.strip().lower())
        if key and key in seen:
            continue
        if key:
            seen.add(key)
        result.append(line)
    
    return '\n'.join(result)


def remove_many_blank_lines(text):
    return re.sub(r'\n{3,}', '\n\n', text)


Run all functions on text

In [ ]:
def clean_text(text):
    text = normalize_line_endings(text)
    text = replace_special_spaces(text)
    text = remove_zero_width_characters(text)
    text = fix_hyphenated_linebreaks(text)
    text = remove_pdf_page_markers(text)
    text = remove_urls_and_emails(text)
    text = remove_trailing_spaces(text)
    text = remove_repeated_spaces(text)
    text = normalize_linebreak_spacing(text)
    text = merge_broken_lines(text)
    text = remove_reference_like_lines(text)
    text = remove_duplicate_lines(text)
    text = remove_many_blank_lines(text)
    return text.strip()


Run sanity checks

In [ ]:
def sanity_check(text):
    print("Original Text:\n")
    print(text[:500])
    print("\n---\n")
    print("Cleaned Text:\n")
    cleaned = clean_text(text)
    print(cleaned[:500])
    return cleaned

print(sanity_check(project_data['raw_text'][0]))

Original Text:

 
 
  
HZ University of Applied Sciences 
Data Driven Business (CU75072V2) 
DATA DRIVEN BUSINESS 
Lars de Bruijne, Joeri Harreman 
 
 01/04/2005 - 04/06/2056  
 
1 | P a g e  
 
Abstract 
This report examines how the magical institution of Hogwarts can become data driven. 
We combine theoretical and practical data analysis. First, this study explores core 
principles of data driven decision making, where we cover data literacy, culture, and 
governance. Then it analyzes the quality of a provided

---

Cleaned Text:

HZ University of Applied Sciences Data Driven Business (CU75072V2) DATA DRIVEN BUSINESS Lars de Bruijne, Joeri Harreman

01/04/2005 - 04/06/2056

Abstract
This report examines how the magical institution of Hogwarts can become data driven.
We combine theoretical and practical data analysis. First, this study explores core principles of data driven decision making, where we cover data literacy, culture, and governance. Then it analyzes the quality of a provid

In [90]:
# save the cleaned data to a new json file
project_data['cleaned_text'] = project_data['raw_text'].apply(clean_text)
project_data.to_json("../../Data/silver/doc_01_silver.json", orient='records', lines=True)